# Lab 3

**Problem.** n children, each with c boolean characteristics. The difference between two children is how many characteristics they differ. They stand in a circle holding hands.We need to find the order of children that maximises the sum of the differences between adjacent children.

## 1. representation

A solution is a permutation of the list of children. We have no illegal solution, because with permutation every child is present exactly once.

The same circle can be written as a list in 2n ways (n rotations * 2 reading directions).
We keep only one: 
- smallest child comes first (rotation)
- of its neighbours, the right one should be smaller, and the left one bigger, if not the circle is read backwards. 

Names are used to order the children.

In [1]:
from collections.abc import Sequence   
from random import sample, random, choices, seed  
from itertools import permutations
from functools import total_ordering, cache    
from zlib import crc32

NUM_CHILDREN = 30
NUM_CHARATECTERISTICS = 20
SEED = 32

In [2]:
@total_ordering
class Child:
    _name: str
    _characteristics: tuple[bool, ...]

    def __init__(self, name: str, characteristics: Sequence[bool]):
        self._name = name
        self._characteristics = tuple(characteristics)

    def difference(self, other: "Child"):
        return sum(s != o for s, o in zip(self._characteristics, other._characteristics))

    def __str__(self) -> str:
        characteristics = "".join(
            "+" if c else "-"
            for c in self._characteristics
        )
        return f"{self._name}: {characteristics}"

    def __repr__(self) -> str:
        return self._name

    def __eq__(self, other):
        return self._name == other._name

    def __lt__(self, other):
        return self._name < other._name

    def __hash__(self):
        return crc32(self._name.encode())


seed(SEED)
children = list()
for c in range(NUM_CHILDREN):
    name = f"Child {c}"
    children.append(Child(name, sample([True, False], k=NUM_CHARATECTERISTICS, counts=[NUM_CHARATECTERISTICS, NUM_CHARATECTERISTICS])))

for child in children[:3]:
    print(child)
print("difference between Child 0 and Child 1:", children[0].difference(children[1]))

Child 0: +++++-+++++------+-+
Child 1: -+-+-++--+++++--++-+
Child 2: ++--++-+-++---++---+
difference between Child 0 and Child 1: 10


## 2. fitness

The sum of the differences between adjacent children, each pair counted once.

In [3]:
@cache
def difference(a: Child, b: Child) -> int:
    return a.difference(b)


def fitness(solution: Sequence[Child]) -> int:
    sol = list(solution)
    return sum(difference(c, cn) for c, cn in zip(sol, sol[1:] + sol[:1]))

## 3. canonical form and mutation

- swap: swap two children
- inversion: reverse a segment of the circle

In [11]:
def normalize(solution: Sequence[Child]) -> list[Child]:
    sol = list(solution)
    i = sol.index(min(sol))
    sol = sol[i:] + sol[:i]                   # smallest child first (rotation)
    if sol[1] > sol[-1]:                      # right > left
        sol = [sol[0]] + sol[:0:-1]           # read backwards
    return sol


def random_solution(children: Sequence[Child]) -> list[Child]:
    return normalize(sample(children, k=len(children)))


def tweak_swap(solution: Sequence[Child]) -> list[Child]:   # swap
    c1, c2 = sample(range(len(solution)), k=2)
    new_solution = list(solution)
    new_solution[c1], new_solution[c2] = solution[c2], solution[c1]
    return normalize(new_solution)


def tweak_inversion(solution: Sequence[Child]) -> list[Child]:
    i, j = sorted(sample(range(len(solution)), k=2))
    new_solution = list(solution)
    new_solution[i:j + 1] = reversed(new_solution[i:j + 1])
    return normalize(new_solution)


# check that normalize works
a = sample(children[:6], k=6) #random circle
b = a[::-1] # same circle read backwards
print("Solution A: " + str(a))
print("Solution B: " + str(b))
print("Normalized A: " + str(normalize(a)))
print("Normalized B: " + str(normalize(b)))

Solution A: [Child 2, Child 4, Child 1, Child 3, Child 0, Child 5]
Solution B: [Child 5, Child 0, Child 3, Child 1, Child 4, Child 2]
Normalized A: [Child 0, Child 3, Child 1, Child 4, Child 2, Child 5]
Normalized B: [Child 0, Child 3, Child 1, Child 4, Child 2, Child 5]


## 4. order crossover (OX)

Normal crossover of two permutations would repeat some children and lose others, obtaining illegal solutions. Order crossover keeps a random segment of parent 1 in place and fills the other positions with the remaining children in the order in which they appear in parent 2.

In [9]:
def order_crossover(p1, p2):
    i, j = sorted(sample(range(len(p1)), k=2))
    segment = list(p1[i:j])
    taken = set(segment)
    rest = []
    for c in p2:
        if c not in taken:
            rest.append(c)
    return normalize(rest[:i] + segment + rest[i:])

## 5. baselines

- random: a random circle
- greedy: start from Child 0 and always put next the child that differs most from the last one (deterministic)
- hill climbing: start from one solution, tweak it, keep the new one if it is not worse (>= to allow moving on a plateu).
  It stops when it goes over the budget of evaluations, or with optional patience (no improvement for x consecutive evaluations).

In [ ]:
def greedy(children):
    sol, free = [children[0]], set(children[1:])
    while free:
        next = max(free, key=lambda c: difference(sol[-1], c))
        sol.append(next)
        free.remove(next)
    return normalize(sol)


def hill_climbing(children, tweak_function=tweak_inversion, max_evals=20_000,
                  patience=None, trace=None):
    current = random_solution(children)
    current_f = fitness(current)
    evals = 1
    last_improvement = 1
    history = [current_f]
    while evals < max_evals:
        candidate = tweak_function(current)
        candidate_f = fitness(candidate)
        evals += 1
        if trace is not None:
            trace.append((evals, candidate_f))
        if candidate_f > current_f:
            last_improvement = evals
        if candidate_f >= current_f:
            current, current_f = candidate, candidate_f
        history.append(current_f)
        if patience is not None and evals - last_improvement >= patience:
            break
    return current, current_f, history

## 6. evolutionary algorithm

- tournament: we take tau individuals from the population uniformly with replacement, best wins.
- hypermodern cycle: to create new individuals, first we chose the operator, then we need to select as many parents as the operator needs, and then we apply it.
- mu + lambda: parents and offspring compete together and the best `mu` survive. 

In [ ]:
def tournament(population, fitnesses, tau=3):
    contenders = choices(range(len(population)), k=tau)
    return population[max(contenders, key=lambda i: fitnesses[i])]


def survivors(population, fitnesses, offspring, offspring_f, mu):
    pool = {}
    for s, f in zip(population + offspring, fitnesses + offspring_f):
        pool[tuple(s)] = f
    best = sorted(pool.items(), key=lambda item: item[1], reverse=True)[:mu]
    return [list(s) for s, _ in best], [f for _, f in best]


OPERATORS = [(order_crossover, 2, 0.4), (tweak_inversion, 1, 0.4), (tweak_swap, 1, 0.2)]


def evolutionary_algorithm(children, max_evals=20_000, mu=50, lam=30, tau=3,
                           trace=None):
    functions = [op for op, _, _ in OPERATORS]
    weights = [w for _, _, w in OPERATORS]
    parents_needed = {op: n for op, n, _ in OPERATORS}
    population = [random_solution(children) for _ in range(mu)]
    fitnesses = [fitness(s) for s in population]
    evals = mu
    history = [max(fitnesses)] * mu
    if trace is not None:
        trace["offspring"] = []
        trace["average"] = [(evals, sum(fitnesses) / mu)]
    while evals + lam <= max_evals:
        offspring = []
        for _ in range(lam):
            op = choices(functions, weights=weights)[0]
            parents = [tournament(population, fitnesses, tau)
                       for _ in range(parents_needed[op])]
            offspring.append(op(*parents))
        offspring_f = [fitness(s) for s in offspring]
        if trace is not None:
            trace["offspring"] += [(evals + k + 1, f)
                                   for k, f in enumerate(offspring_f)]
        evals += lam
        population, fitnesses = survivors(population, fitnesses, offspring, offspring_f, mu)
        history += [fitnesses[0]] * lam
        if trace is not None:
            trace["average"].append((evals, sum(fitnesses) / len(fitnesses)))
    return population[0], fitnesses[0], history

## 7. results 

n = 30 children, 20 characteristics

First one run of each, to see what happens (stopping the hill climber on stagnation), then the circle found.

In [ ]:
seed(SEED)
print("random circle:", fitness(random_solution(children)), "| greedy:", fitness(greedy(children)))

trace_hc, trace_ea = [], {}
hc_best, hc_f, hc_history = hill_climbing(children, tweak_inversion, 20_000, patience=2_000, trace=trace_hc)
print(f"hill climbing, stop on stagnation (patience 2000): fitness {hc_f} after {len(hc_history)} evaluations")
_, _, hc_budget_history = hill_climbing(children, tweak_inversion, 20_000)
print(f"hill climbing, full budget of 20,000 evaluations:  fitness {hc_budget_history[-1]}")

ea_best, ea_f, ea_history = evolutionary_algorithm(children, trace=trace_ea)
print(f"evolutionary algorithm (hypermodern), 20,000 evaluations: fitness {ea_f}")
print("average fitness of the final population:", round(trace_ea["average"][-1][1], 1))
print()
print("the circle found by the evolutionary algorithm:")
print(" -> ".join(c._name for c in ea_best) + f" -> (back to {ea_best[0]._name})")

### Every candidate, the best solution and the population average
Grey dots: the fitness of every candidate that was evaluated. Red: best (current) solution. Blue dashed:
average fitness of the population. When the blue line reaches the red one the population is (almost) all
the same solution: diversity is lost.

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    left.scatter([e for e, _ in trace_hc], [f for _, f in trace_hc], s=3, color="gray", alpha=0.3, label="candidates")
    left.plot(range(1, len(hc_history) + 1), hc_history, color="red", linewidth=2, label="current solution")
    left.set_title("Hill climbing (inversion, stop on stagnation)")
    offspring = trace_ea["offspring"]
    right.scatter([e for e, _ in offspring], [f for _, f in offspring], s=3, color="gray", alpha=0.3, label="offspring")
    right.plot(range(1, len(ea_history) + 1), ea_history, color="red", linewidth=2, label="best")
    right.plot([e for e, _ in trace_ea["average"]], [a for _, a in trace_ea["average"]], color="blue",
               linestyle="--", label="population average")
    right.set_title("Evolutionary algorithm (hypermodern)")
    for ax in (left, right):
        ax.set_xlabel("fitness evaluations"); ax.grid(alpha=0.3); ax.legend(loc="lower right")
    left.set_ylabel("fitness")
    plt.tight_layout(); plt.savefig("lab3_essenziale.png", dpi=110); plt.show()
except ImportError:
    print("matplotlib not installed: pip install matplotlib to see the plot")

### A fair comparison
The same budget (20,000 evaluations) for everybody, 10 independent runs, mean of the final fitness.
(With a stop on stagnation every algorithm would use a different number of evaluations and the comparison
would not be fair.)

In [ ]:
def mean(values):
    return sum(values) / len(values)

runs = 10
results = {
    "random circle (mean of 200)": mean([fitness(random_solution(children)) for _ in range(200)]),
    "greedy": fitness(greedy(children)),
    "hill climbing, swap": mean([hill_climbing(children, tweak_swap)[1] for _ in range(runs)]),
    "hill climbing, inversion": mean([hill_climbing(children, tweak_inversion)[1] for _ in range(runs)]),
    "evolutionary algorithm": mean([evolutionary_algorithm(children)[1] for _ in range(runs)]),
}
for name, value in results.items():
    print(f"{name:30s} {value:8.1f}")